# Intervention Effects

Measures model response to interventions: activation scaling, direction addition, component knockouts, transferability, and multi-layer robustness.

**References:**
- Turner et al. (2023) "Activation Addition: Steering Language Models"
- Li et al. (2023) "Inference-Time Intervention"

## Why This Matters

Intervention effect analysis measures the downstream consequences of modifying activations — scaling sensitivity, direction sweeps, knockout recovery, and transferability. This characterizes the causal structure of the network: not just which components matter, but how sensitive behavior is to the magnitude and direction of changes.

**Key references:**
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations
- [Li et al. (2023) "Inference-Time Intervention"](https://arxiv.org/abs/2306.03341) — Targeted activation interventions for truthfulness

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.intervention_effects import (
    activation_scaling_sensitivity,
    direction_addition_sweep,
    component_knockout_recovery,
    intervention_transferability,
    multi_layer_knockout,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers")

In [ ]:
# 1. Activation scaling sensitivity
for l in range(cfg.n_layers):
    ss = activation_scaling_sensitivity(model, tokens, metric, layer=l)
    print(f"Layer {l}: sensitivity={ss['sensitivity']:.4f}, monotonic={ss['monotonic']}")
    print(f"  scales: {[f'{s:.1f}' for s in ss['scales']]}")
    print(f"  metrics: {[f'{m:.4f}' for m in ss['metrics']]}")

In [ ]:
# 2. Direction addition sweep
rng = np.random.RandomState(42)
direction = rng.randn(32).astype(np.float32)
direction /= np.linalg.norm(direction)
da = direction_addition_sweep(model, tokens, direction, metric, layer=1)
print(f"Baseline: {da['baseline_metric']:.4f}")
print(f"Optimal coefficient: {da['optimal_coefficient']:.2f}")
print(f"Effect range: {da['effect_range']:.4f}")
for c, m in zip(da['coefficients'], da['metrics']):
    print(f"  alpha={c:+.1f}: metric={m:.4f}")

In [ ]:
# 3. Component knockout recovery
ko = component_knockout_recovery(model, tokens, metric)
print(f"Baseline: {ko['baseline_metric']:.4f}")
print(f"Most essential: {ko['most_essential']}")
print(f"Most self-sufficient: {ko['most_self_sufficient']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: attn_ko={ko['knockout_effects'][l,0]:.4f}, "
          f"mlp_ko={ko['knockout_effects'][l,1]:.4f}, "
          f"attn_solo={ko['solo_metrics'][l,0]:.4f}, "
          f"mlp_solo={ko['solo_metrics'][l,1]:.4f}")

In [ ]:
# 4. Multi-layer knockout
mlk = multi_layer_knockout(model, tokens, metric)
print(f"Baseline: {mlk['baseline_metric']:.4f}")
print(f"Half-perf threshold: {mlk['half_performance_threshold']} layers")
print(f"Graceful degradation: {mlk['graceful_degradation']}")
for n, m in zip(mlk['n_knocked_out'], mlk['metrics']):
    print(f"  {n} knocked out: metric={m:.4f}")